# Candidate's Choice — Pipeline Breakdown
## "Where does the delay actually happen?"

**Business justification:** Telling the CEO that deliveries are late is not actionable. Knowing *which phase* of the pipeline is responsible tells Veridi exactly where to invest:
- If the delay is in **transit** → negotiate carrier SLAs or switch logistics providers
- If the delay is in the **warehouse** → fix internal fulfilment operations

**The 3 pipeline phases:**
```
[Purchase] ──► [Approved] ──► [Carrier Pickup] ──► [Delivered to Customer]
        Processing        Warehouse → Carrier        Transit
```

**Input:** `../data/02_delivered.parquet`  
**Output:** `../data/06_pipeline.parquet`

---

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

ORDER_CATS = ['On Time', 'Late', 'Super Late']
print('Libraries loaded.')

## Step 1 — Load data

In [ ]:
delivered = pd.read_parquet('../data/02_delivered.parquet')

# Convert date cols back to datetime (parquet preserves types, just confirming)
for col in ['order_purchase_timestamp','order_approved_at',
            'order_delivered_carrier_date','order_delivered_customer_date']:
    delivered[col] = pd.to_datetime(delivered[col])

print(f'Loaded: {len(delivered):,} delivered orders')

## Step 2 — Calculate the three pipeline phases

In [ ]:
pipeline = delivered.dropna(subset=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date'
]).copy()

pipeline['processing_days'] = (
    pipeline['order_approved_at'] - pipeline['order_purchase_timestamp']
).dt.total_seconds() / 86400

pipeline['warehouse_days'] = (
    pipeline['order_delivered_carrier_date'] - pipeline['order_approved_at']
).dt.total_seconds() / 86400

pipeline['transit_days'] = (
    pipeline['order_delivered_customer_date'] - pipeline['order_delivered_carrier_date']
).dt.total_seconds() / 86400

# Remove any implausible negative durations (data quality)
before = len(pipeline)
for col in ['processing_days', 'warehouse_days', 'transit_days']:
    pipeline = pipeline[pipeline[col] >= 0]
after = len(pipeline)
print(f'Rows after removing negative durations: {after:,}  (removed {before - after:,})')

## Step 3 — Compare phases by delivery status

In [ ]:
phase_by_status = (
    pipeline
    .groupby('delivery_status')[['processing_days', 'warehouse_days', 'transit_days']]
    .mean()
    .reindex(ORDER_CATS)
    .round(2)
)

phase_by_status.columns = ['Payment Processing', 'Warehouse → Carrier', 'Carrier → Customer (Transit)']
print('Average phase duration (days) by delivery status:')
print(phase_by_status.to_string())

## Step 4 — Stacked bar chart

In [ ]:
phase_melt = (
    phase_by_status
    .reset_index()
    .melt(id_vars='delivery_status', var_name='Phase', value_name='Days')
)

fig = px.bar(
    phase_melt,
    x='delivery_status', y='Days', color='Phase',
    barmode='stack',
    color_discrete_sequence=['#3498db', '#9b59b6', '#e67e22'],
    title='Where Does the Time Go? — Average Pipeline Phases by Delivery Status',
    labels={'delivery_status': 'Delivery Status', 'Days': 'Average Days'},
    category_orders={'delivery_status': ORDER_CATS},
    text_auto='.1f',
)
fig.update_traces(textposition='inside', insidetextanchor='middle')
fig.update_layout(legend_title='Pipeline Phase')
fig.show()

## Step 5 — Identify the dominant delay driver

In [ ]:
ontime  = phase_by_status.loc['On Time']
slat    = phase_by_status.loc['Super Late']

transit_delta   = slat['Carrier → Customer (Transit)']   - ontime['Carrier → Customer (Transit)']
warehouse_delta = slat['Warehouse → Carrier']             - ontime['Warehouse → Carrier']
processing_delta= slat['Payment Processing']              - ontime['Payment Processing']

print('Extra days added per phase for Super Late vs On Time orders:')
print(f'  Payment Processing:          +{processing_delta:.1f} days')
print(f'  Warehouse → Carrier:         +{warehouse_delta:.1f} days')
print(f'  Carrier → Customer (Transit):+{transit_delta:.1f} days')

dominant = max(
    [('Payment Processing', processing_delta),
     ('Warehouse → Carrier', warehouse_delta),
     ('Carrier → Customer (Transit)', transit_delta)],
    key=lambda x: x[1]
)

print(f'\n>>> Primary delay driver: {dominant[0]}  (+{dominant[1]:.1f} days)')

if dominant[0] == 'Carrier → Customer (Transit)':
    action = 'Negotiate carrier SLAs or explore alternative logistics providers for high-delay routes.'
elif dominant[0] == 'Warehouse → Carrier':
    action = 'Audit internal order fulfilment — reduce warehouse dwell time before handoff to carrier.'
else:
    action = 'Investigate payment approval delays — may indicate fraud checks or payment failures.'

print(f'>>> Recommended action: {action}')

## Step 6 — Save

In [ ]:
pipeline_cols = [
    'order_id', 'customer_state', 'delivery_status', 'delivery_delay_days',
    'review_score', 'processing_days', 'warehouse_days', 'transit_days'
]
pipeline[pipeline_cols].to_parquet('../data/06_pipeline.parquet', index=False)
print('Saved → ../data/06_pipeline.parquet')
print('\n✅ All 6 notebooks complete! The data is ready for the Streamlit dashboard.')